# Movie Recommendation System
This notebook implements a recommendation system using Singular Value Decomposition. It filters the dataset to active users and popular movies to reduce memory usage.

In [13]:
import pandas as pd
import numpy as np
from scipy.sparse.linalg import svds
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
Loading the filtered dataset (`ml-32m-filtered`).

In [14]:
ratings_path = '/content/ratings.csv'
movies_path = '/content/movies.csv'

movies_df = pd.read_csv(movies_path)
ratings_df = pd.read_csv(ratings_path)

print(f"Original Ratings shape: {ratings_df.shape}")
print(f"Original Movies shape: {movies_df.shape}")

Original Ratings shape: (4821916, 4)
Original Movies shape: (35958, 3)


## 2. Filter Dataset to Reduce Memory
Filtering to users who rated at least 50 movies and movies that have at least 50 ratings.

In [15]:
min_movie_ratings = 50
movie_counts = ratings_df['movieId'].value_counts()
filter_movies = movie_counts[movie_counts > min_movie_ratings].index.tolist()

min_user_ratings = 50
user_counts = ratings_df['userId'].value_counts()
filter_users = user_counts[user_counts > min_user_ratings].index.tolist()

ratings_df_filtered = ratings_df[
    (ratings_df['movieId'].isin(filter_movies)) &
    (ratings_df['userId'].isin(filter_users))
]
print(f"Filtered Ratings shape: {ratings_df_filtered.shape}")

Filtered Ratings shape: (3566718, 4)


## 3. Create User-Item Matrix
Pivot the table. NaNs are filled with 0.

In [16]:
# Create pivot table
user_item_matrix = ratings_df_filtered.pivot(index='userId', columns='movieId', values='rating')


users = user_item_matrix.index.tolist()
movies = user_item_matrix.columns.tolist()

# Fill Null values with 0
user_item_matrix = user_item_matrix.fillna(0)
print(f"User-Item Matrix shape: {user_item_matrix.shape}")
user_item_matrix.head()

User-Item Matrix shape: (23664, 4753)


movieId,73268,73319,73321,73929,74131,74154,74156,74450,74452,74458,...,288669,288679,288861,288979,289097,289253,289295,289297,290213,290407
userId,,,,,,,,,,,,,,,,,,,,,
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.5,3.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,3.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
46,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,4.5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Normalization & SVD
Normalize by subtracting the user mean, then apply SVD.

In [17]:
R = user_item_matrix.values
user_ratings_mean = np.mean(R, axis=1)
R_demeaned = R - user_ratings_mean.reshape(-1, 1)

# Apply SVD. We choose k=50 latent features
k = min(50, min(R.shape)-1)
U, sigma, Vt = svds(R_demeaned, k=k)

sigma = np.diag(sigma)
print("SVD completed.")

SVD completed.


## 5. Reconstruct the Matrix
Dot product of U, Sigma, and Vt, then add back the user means.

In [18]:
all_user_predicted_ratings = np.dot(np.dot(U, sigma), Vt) + user_ratings_mean.reshape(-1, 1)

# Create a DataFrame for predictions
preds_df = pd.DataFrame(all_user_predicted_ratings, columns=movies, index=users)
print(f"Predictions Matrix shape: {preds_df.shape}")
preds_df.head()

Predictions Matrix shape: (23664, 4753)


,73268,73319,73321,73929,74131,74154,74156,74450,74452,74458,...,288669,288679,288861,288979,289097,289253,289295,289297,290213,290407
10,0.659770,0.004812,1.433881,0.444391,-0.010424,0.067822,0.142152,0.094000,0.266554,3.489769,...,0.008415,-0.012162,-0.012158,0.003021,-0.019479,-0.023572,0.001426,-0.007580,-0.016295,-0.017684
16,0.049624,0.325919,1.672000,0.111816,0.017556,0.208668,0.232383,0.206438,-0.117663,3.538103,...,-0.017005,0.012574,-0.033198,-0.021545,-0.027191,-0.053957,0.001549,0.025459,0.023543,-0.030270
28,0.974197,0.304946,2.087779,0.254385,0.072487,0.048292,0.259534,0.268148,0.236016,2.198118,...,-0.041764,-0.013504,0.085320,0.040365,0.063974,0.146403,-0.000687,0.022766,0.020612,-0.000640
35,0.121066,0.327913,-0.178937,-0.007667,-0.031918,0.223153,0.023983,0.165336,-0.005218,2.380344,...,-0.019539,-0.024335,-0.024105,-0.033807,-0.033561,-0.056466,-0.029745,-0.020117,-0.026860,-0.004738
46,0.214903,0.165784,0.634968,-0.055009,0.016077,0.036259,-0.062644,0.166847,0.001430,4.218498,...,-0.007639,-0.000795,0.001851,-0.004379,0.005571,0.026415,-0.009102,0.000881,-0.005656,-0.005216


## 6. Recommendation Logic
Function to recommend top 5 unrated movies for a given user.

In [19]:
def recommend_movies(user_id, num_recommendations=5):
    if user_id not in preds_df.index:
        return f"User {user_id} not found in the filtered dataset. Try a different user."

    # to find unrated movies
    user_row = user_item_matrix.loc[user_id]
    unrated_movies = user_row[user_row == 0].index

    # Get the predicted ratings for this user
    user_preds = preds_df.loc[user_id]

    # Filter predictions to only include unrated movies
    unrated_preds = user_preds[unrated_movies]

    # Sort by highest ratings
    top_movie_ids = unrated_preds.sort_values(ascending=False).head(num_recommendations).index

    recommendations = movies_df[movies_df['movieId'].isin(top_movie_ids)][['movieId', 'title', 'genres']]
    recommendations = recommendations.set_index('movieId').loc[top_movie_ids].reset_index()

    return recommendations

## 7. Get Recommendations
Interactively prompt for a user ID and print recommendations.

In [28]:
# Example usage
sample_user = users[35]
print(f"Recommending for user {sample_user}:")
display(recommend_movies(sample_user, 5))

Recommending for user 273:


,movieId,title,genres
0,97913,Wreck-It Ralph (2012),Animation|Comedy
1,110102,Captain America: The Winter Soldier (2014),Action|Adventure|Sci-Fi|IMAX
2,88140,Captain America: The First Avenger (2011),Action|Adventure|Sci-Fi|Thriller|War
3,103335,Despicable Me 2 (2013),Animation|Children|Comedy|IMAX
4,102125,Iron Man 3 (2013),Action|Sci-Fi|Thriller|IMAX
